## Step 1: Load and Parse Test Data

First, let's read the data from output_data.txt. The file has space-separated numbers that we need to parse into arrays.

# Kernel Function Test for Vacuum_vac.jl

This notebook tests the `kernel!` function from `Vacuum_vac.jl` using test data from `output_data.txt`.

## Test Setup

The output_data.txt file contains space-separated numerical data that appears to be organized in rows. We'll:
1. Load the JPEC module
2. Parse the input data from output_data.txt
3. Set up the test parameters for the kernel function
4. Run the kernel function and compare results

In [1]:
# Load required packages
using DelimitedFiles
using LinearAlgebra

# Add JPEC to path and load
push!(LOAD_PATH, joinpath(@__DIR__, "../.."))
using JPEC


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
# Read the output_data.txt file
data_file = joinpath(@__DIR__, "output_data.txt")

println("Reading file: $data_file")
println("File exists: $(isfile(data_file))")
println()

# Read all lines
lines = readlines(data_file)
println("Total lines in file: $(length(lines))")

# Helper function to parse a line of comma-separated numbers
function parse_number_line(line)
    tokens = split(line, ',')
    return [parse(Float64, strip(token)) for token in tokens if !isempty(strip(token))]
end

# Parse the five lines
if length(lines) < 5
    error("Expected 5 lines in output_data.txt, but found only $(length(lines)) lines!")
end

println("\nParsing data from first 5 lines:")
xobs = parse_number_line(lines[1])
println("  Line 1 (xobs): $(length(xobs)) values")

zobs = parse_number_line(lines[2])
println("  Line 2 (zobs): $(length(zobs)) values")

xsce = parse_number_line(lines[3])
println("  Line 3 (xsce): $(length(xsce)) values")

zsce = parse_number_line(lines[4])
println("  Line 4 (zsce): $(length(zsce)) values")

params = parse_number_line(lines[5])
println("  Line 5 (params): $(length(params)) values")

println("\n✓ Successfully parsed all data!")
println("\nData summary:")
println("  xobs: $(length(xobs)) points, range [$(minimum(xobs)), $(maximum(xobs))]")
println("  zobs: $(length(zobs)) points, range [$(minimum(zobs)), $(maximum(zobs))]")
println("  xsce: $(length(xsce)) points, range [$(minimum(xsce)), $(maximum(xsce))]")
println("  zsce: $(length(zsce)) points, range [$(minimum(zsce)), $(maximum(zsce))]")
println("  params: $(params)")

# Extract kernel function parameters
j1_input = Int(params[1])
j2_input = Int(params[2])
isgn_input = Int(params[3])
iopw_input = Int(params[4])
iops_input = Int(params[5])
wall_flag_input = params[6] != 0  # Convert to boolean

println("\nKernel parameters:")
println("  j1 = $j1_input")
println("  j2 = $j2_input")
println("  isgn = $isgn_input")
println("  iopw = $iopw_input")
println("  iops = $iops_input")
println("  wall_flag = $wall_flag_input")

Reading file: /Users/priyansh/Documents/Git/JPEC/notebooks/vacuum_tests/output_data.txt
File exists: true

Total lines in file: 5

Parsing data from first 5 lines:
  Line 1 (xobs): 517 values
  Line 2 (zobs): 517 values
  Line 3 (xsce): 517 values
  Line 4 (zsce): 517 values
  Line 5 (params): 6 values

✓ Successfully parsed all data!

Data summary:
  xobs: 517 points, range [0.0, 2.2891936601349068]
  zobs: 517 points, range [-1.1134694034127417, 0.9616419248701459]
  xsce: 517 points, range [0.0, 2.2891936601349068]
  zsce: 517 points, range [-1.1134694034127417, 0.9616419248701459]
  params: [1.0, 1.0, -1.0, 1.0, 1.0, 0.0]

Kernel parameters:
  j1 = 1
  j2 = 1
  isgn = -1
  iopw = 1
  iops = 1
  wall_flag = false


## Step 2: Set Up Kernel Function Parameters

Data has been loaded and kernel parameters extracted from the input file.

In [3]:
# Set up parameters for kernel function test
nobs = length(xobs)
nsrc = length(xsce)

# Validate data consistency
if length(xobs) != length(zobs)
    error("xobs and zobs have different lengths: $(length(xobs)) vs $(length(zobs))")
end
if length(xsce) != length(zsce)
    error("xsce and zsce have different lengths: $(length(xsce)) vs $(length(zsce))")
end

println("Data validation passed!")
println()

# Initialize output matrices
grdgre = zeros(Float64, nobs, nsrc)
gren = zeros(Float64, nobs, nsrc)

# Use parameters loaded from input file
j1 = j1_input
j2 = j2_input
isgn = isgn_input
iopw = iopw_input
iops = iops_input
wall_flag = wall_flag_input

println("Test configuration:")
println("  Observer points: $nobs")
println("  Source points: $nsrc")
println("  Observer type (j1): $j1 (1=plasma, 2=wall)")
println("  Source type (j2): $j2 (1=plasma, 2=wall)")
println("  Sign parameter (isgn): $isgn")
println("  Wall option (iopw): $iopw")
println("  Log singularity option (iops): $iops")
println("  Wall flag: $wall_flag")
println()
println("Output matrix dimensions:")
println("  grdgre: $(size(grdgre))")
println("  gren: $(size(gren))")

Data validation passed!

Test configuration:
  Observer points: 517
  Source points: 517
  Observer type (j1): 1 (1=plasma, 2=wall)
  Source type (j2): 1 (1=plasma, 2=wall)
  Sign parameter (isgn): -1
  Wall option (iopw): 1
  Log singularity option (iops): 1
  Wall flag: false

Output matrix dimensions:
  grdgre: (517, 517)
  gren: (517, 517)


## Step 3: Run the Kernel Function

The `kernel!` function signature from Vacuum_vac.jl:
```julia
kernel!(grdgre, gren, xobs, zobs, xsce, zsce, j1, j2, isgn, iopw, iops, wall_flag)
```

Now we'll call the kernel function with our test data.

## Step 4: Analyze Results

Visualize and analyze the kernel function output.

In [12]:
# Run the kernel function
# Note: This may fail if additional setup/initialization is required
try
    JPEC.VacuumMod.kernel!(
        grdgre, gren, 
        xobs, zobs, 
        xsce, zsce, 
        j1, j2, isgn, iopw, iops, wall_flag
    )
    
    println("✓ Kernel function executed successfully!")
    println()
    println("Result statistics:")
    println("  grdgre: min=$(minimum(grdgre)), max=$(maximum(grdgre)), mean=$(sum(grdgre)/length(grdgre))")
    println("  gren: min=$(minimum(gren)), max=$(maximum(gren)), mean=$(sum(gren)/length(gren))")
    
catch e
    println("✗ Error running kernel function:")
    println(e)
    println()
    println("Stack trace:")
    for (exc, bt) in Base.catch_stack()
        showerror(stdout, exc, bt)
        println()
    end
    println()
end

In [10]:
# Basic analysis of results
if !iszero(grdgre) || !iszero(gren)
    println("Non-zero elements in grdgre: $(count(!iszero, grdgre))")
    println("Non-zero elements in gren: $(count(!iszero, gren))")
    println()
    
    # Check for NaN or Inf values
    println("NaN values in grdgre: $(count(isnan, grdgre))")
    println("NaN values in gren: $(count(isnan, gren))")
    println("Inf values in grdgre: $(count(isinf, grdgre))")
    println("Inf values in gren: $(count(isinf, gren))")
    
    # Sample some values
    println()
    println("Sample values from grdgre (first 5x5):")
    display(grdgre[1:min(5,size(grdgre,1)), 1:min(5,size(grdgre,2))])
    println()
    println("Sample values from gren (first 5x5):")
    display(gren[1:min(5,size(gren,1)), 1:min(5,size(gren,2))])
else
    println("Results are all zeros - kernel function may not have run successfully")
end

In [14]:
# Add a simple test
test_grdgre = zeros(Float64, 2, 2)
test_gren = zeros(Float64, 2, 2)
test_xobs = [2.0, 2.1]
test_zobs = [0.0, 0.1]
test_xsce = [2.05, 2.15]
test_zsce = [0.05, 0.15]

println("Before kernel!: grdgre sum = $(sum(test_grdgre))")
kernel!(test_grdgre, test_gren, test_xobs, test_zobs, test_xsce, test_zsce, 
        1, 1, -1, 1, 1, false)
println("After kernel!: grdgre sum = $(sum(test_grdgre))")
println("After kernel!: gren sum = $(sum(test_gren))")


UndefVarError: UndefVarError: `kernel!` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
# Debug: Check what's in the result matrices
println("===== RESULT DIAGNOSTICS =====")
println("grdgre exists: $(isdefined(Main, :grdgre))")
println("gren exists: $(isdefined(Main, :gren))")
println()

if isdefined(Main, :grdgre) && isdefined(Main, :gren)
    println("grdgre size: $(size(grdgre))")
    println("gren size: $(size(gren))")
    println()
    
    println("grdgre all zeros: $(all(grdgre .== 0))")
    println("gren all zeros: $(all(gren .== 0))")
    println()
    
    println("grdgre sum: $(sum(grdgre))")
    println("gren sum: $(sum(gren))")
    println()
    
    println("grdgre first 3x3:")
    display(grdgre[1:min(3,size(grdgre,1)), 1:min(3,size(grdgre,2))])
    println()
    
    println("gren first 3x3:")
    display(gren[1:min(3,size(gren,1)), 1:min(3,size(gren,2))])
else
    println("ERROR: Result matrices not defined!")
    println("Make sure to run the kernel execution cell first.")
end

ArgumentError: ArgumentError: isdefined: too few arguments (expected 2)